<a href="https://colab.research.google.com/github/1saacbp/Tesis-IBP-Texto-to-SQL/blob/main/Limpieza_CargaBD_FineTunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
datos=pd.read_csv('/content/Tasas_activas_por_credito_Historico_20250807.csv')

In [7]:
import unicodedata

def remove_accents(input_str):
    if isinstance(input_str, str):
        nfkd_form = unicodedata.normalize('NFKD', input_str)
        return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])
    return input_str

text_columns = datos.select_dtypes(include='object').columns

for col in text_columns:
    datos[col] = datos[col].apply(lambda x: remove_accents(str(x).lower()) if pd.notna(x) else x)


In [8]:
columnas_deudor = [
    'Tipo_de_persona','Sexo','Clase_deudor',
    'Codigo_Municipio','Codigo_CIIU',
    'Grupo_Etnico','Antiguedad_de_la_empresa',
    'Tamaño_de_empresa'
]

datos= datos.dropna(subset=columnas_deudor)

In [9]:
Entidades = datos[['Tipo_Entidad', 'Nombre_Tipo_Entidad','Codigo_Entidad','Nombre_Entidad']].drop_duplicates().reset_index(drop=True)
Entidades['id_entidad'] = Entidades.index + 1
Deudor = datos[['Tipo_de_persona','Sexo','Clase_deudor','Codigo_Municipio','Codigo_CIIU','Grupo_Etnico','Antiguedad_de_la_empresa','Tamaño_de_empresa']].drop_duplicates().reset_index(drop=True)
Deudor['id_deudor'] = Deudor.index + 1

In [10]:
Credito = datos.merge(
    Entidades,
    on=['Codigo_Entidad','Tipo_Entidad'],
    how='left'
)

In [11]:
Credito=Credito.merge(
    Deudor,
    on=['Tipo_de_persona','Sexo','Clase_deudor','Codigo_Municipio','Codigo_CIIU','Grupo_Etnico','Antiguedad_de_la_empresa','Tamaño_de_empresa'],
    how='left'
)

In [12]:
Credito = Credito.drop(columns=['Tipo_Entidad', 'Nombre_Tipo_Entidad_x','Codigo_Entidad','Nombre_Entidad_x','Nombre_Tipo_Entidad_y','Nombre_Entidad_y',
                              'Tipo_de_persona','Sexo','Clase_deudor','Codigo_Municipio','Codigo_CIIU','Grupo_Etnico','Antiguedad_de_la_empresa','Tamaño_de_empresa',
                              ])


In [13]:
xls = pd.ExcelFile('/content/Tablas_extra_Tesis_IBP.xlsx')
df_ciiu = pd.read_excel(xls, sheet_name='Codigos_CIIU')
df_municipios = pd.read_excel(xls, sheet_name='Codigos_Municipios')


In [14]:
text_columns = df_ciiu.select_dtypes(include='object').columns
text_columnss = df_municipios.select_dtypes(include='object').columns
for col in text_columns:
    df_ciiu[col] = df_ciiu[col].apply(lambda x: remove_accents(str(x).lower()) if pd.notna(x) else x)
for col in text_columnss:
    df_municipios[col] = df_municipios[col].apply(lambda x: remove_accents(str(x).lower()) if pd.notna(x) else x)

In [15]:
df_ciiu = df_ciiu.drop_duplicates(subset=['codigo_ciiu'], keep='last')

In [16]:
Credito = Credito.rename(columns={'Tipo_de_crédito': 'tipo_credito'})
Credito = Credito.rename(columns={'Tipo_de_garantía': 'tipo_garantia'})
Credito = Credito.rename(columns={'Producto de crédito': 'producto_credito'})
Credito = Credito.rename(columns={'Plazo de crédito': 'plazo_credito'})
Credito = Credito.rename(columns={'Tasa_efectiva_promedio_ponderada': 'tasa_efectiva_promedio_ponderada'})
Credito = Credito.rename(columns={'margen_adicional': 'margen_adicional'})
Credito = Credito.rename(columns={'Rango_monto_desembolsado': 'rango_monto_desembolsado'})
Credito = Credito.rename(columns={'Tipo_de_Tasa': 'tipo_tasa'})
Credito = Credito.rename(columns={'Numero_de_creditos_desembolsados': 'numero_creditos_desembolsados'})
Credito = Credito.rename(columns={'Fecha_Corte': 'fecha_corte'})
Credito = Credito.rename(columns={'Montos_desembolsados': 'monto_desembolsado'})
#--------------------------------------------------------------------
Deudor=Deudor.rename(columns={'Codigo_Municipio':'codigo_municipio'})
Deudor=Deudor.rename(columns={'Codigo_CIIU':'codigo_ciiu'})
Deudor=Deudor.rename(columns={'Tipo_de_persona':'tipo_persona'})
Deudor=Deudor.rename(columns={'Sexo':'sexo'})
Deudor=Deudor.rename(columns={'Clase_deudor':'clase_deudor'})
Deudor=Deudor.rename(columns={'Grupo_Etnico':'grupo_etnico'})
Deudor=Deudor.rename(columns={'Antiguedad_de_la_empresa':'antiguedad_empresa'})
Deudor=Deudor.rename(columns={'Tamaño_de_empresa':'tamano_empresa'})
Deudor=Deudor.dropna(subset=["codigo_ciiu"])
#--------------------------------------------------------------------
Entidades=Entidades.rename(columns={'Tipo_Entidad':'tipo_entidad'})
Entidades=Entidades.rename(columns={'Nombre_Tipo_Entidad':'nombre_tipo_entidad'})
Entidades=Entidades.rename(columns={'Codigo_Entidad':'codigo_entidad'})
Entidades=Entidades.rename(columns={'Nombre_Entidad':'nombre_entidad'})

In [ ]:
Deudor['clase_deudor'].value_counts()

,count
clase_deudor,
deudor de la entidad,40277
deudor nuevo en la entidad,33672


In [ ]:
Deudor['grupo_etnico'].value_counts()

,count
grupo_etnico,
sin informacion (1),38376
ningun grupo etnico,10135
no aplica,3681
"negro(a), mulato(a), afrodescendiente, afrocolombiano(a)",327
indigena,245
"raizal del archipielago de san andres, providencia y santa catalina",30
palenquero (a) de san basilio,18
gitano (a) o rrom,5


In [ ]:
Entidades['codigo_entidad'].value_counts()

,count
codigo_entidad,
1,3
2,3
4,2
39,1
43,1
23,1
30,1
54,1
122,1


In [ ]:
Entidades['nombre_entidad'].value_counts()

,count
nombre_entidad,
banagrario,1
rci colombia s.a.,1
banco davivienda,1
banco de bogota,1
cooperativa financiera de antioquia,1
banco de occidente,1
cotrafa,1
banco caja social s.a.,1
bancoomeva,1


In [17]:
Entidades['nombre_tipo_entidad'] = Entidades['nombre_tipo_entidad'].str.replace('bc-establecimiento bancario', 'establecimiento bancario', regex=False)
Entidades['nombre_tipo_entidad'] = Entidades['nombre_tipo_entidad'].str.replace('cf-compania de financiamiento', 'compania de financiamiento', regex=False)

In [ ]:
Entidades['nombre_tipo_entidad'].value_counts()

,count
nombre_tipo_entidad,
establecimiento bancario,27
compania de financiamiento,10
institucion oficial especial,6
cooperativas de caracter financiero,5


In [ ]:
Deudor['tipo_persona'].value_counts()

,count
tipo_persona,
natural,49136
juridica,3681


In [ ]:
Deudor['antiguedad_empresa'].value_counts()

,count
antiguedad_empresa,
no aplica(1),31744
mas de 10 anos,12670
mas de 5 y hasta 10 anos,5259
0 a 5 anos,3144


In [ ]:
Deudor['tamano_empresa'].value_counts()

,count
tamano_empresa,
microempresa,27947
no aplica,21215
pequena empresa,1996
mediana empresa,1011
gran empresa,648


In [ ]:
Credito['tipo_tasa'].value_counts()

,count
tipo_tasa,
fs,506058
ib6,122064
dtf,32213
ibr,9038
ib1,7216
ib3,3786
cec$,207
dtfe,83
ipc,66


In [ ]:
Credito['plazo_credito'].unique()

array(['mas de 5 anos y hasta 7 anos'], dtype=object)

In [18]:
Credito['producto_credito'] = Credito['producto_credito'].str.replace('credito productivo rural  (con recursos de redescuento)', 'productivo rural redescuento', regex=False)
Credito['producto_credito'] = Credito['producto_credito'].str.replace('credito productivo urbano  (con recursos de redescuento)', 'productivo urbano redescuento', regex=False)
Credito['producto_credito'] = Credito['producto_credito'].str.replace('libranza otros', 'libranza', regex=False)

In [ ]:
Credito['producto_credito'].value_counts().head(5)

,count
producto_credito,
libre inversion,116864
libranza,78740
productivo rural redescuento,37959
vehiculo,33817
productivo urbano redescuento,19938


In [19]:
Credito['tipo_garantia'] = Credito['tipo_garantia'].str.replace('garantia del fondo agropecuario de garantias (fag)', 'fag', regex=False)
Credito['tipo_garantia'] = Credito['tipo_garantia'].str.replace('garantia  fondo nacional de garantias (fng) o fondo de garantias de antioquia (fga)', 'fng o fga', regex=False)

In [ ]:
Credito['tipo_garantia'].value_counts()

,count
tipo_garantia,
sin garantia,160609
garantia idonea o no idonea,115700
fag,77022
fng o fga,1673


In [ ]:
Credito['tipo_credito'].value_counts()

,count
tipo_credito,
consumo,251161
credito productivo,77592
comercial ordinario,23132
vivienda,2726
comercial preferencial o corporativo,360
comercial especial,32
microcredito,1


In [ ]:
Credito['rango_monto_desembolsado'].value_counts()

,count
rango_monto_desembolsado,
mayor a 12 smlmv menor o igual a 25 smlmv,122657
mayor a 25 smlmv menor o igual a 50 smlmv,106113
mayor a 6 smlmv menor o igual a 12 smlmv,94909
mayor a 50 smlmv menor o igual a 100 smlmv,64269
mayor a 3 smlmv menor o igual a 6 smlmv,54073
mayor a 12 smlmv y menor o igual a 25 smlmv,53121
mayor a 6 smlmv y menor o igual a 12 smlmv,50279
mayor a 1 smlmv menor o igual a 3 smlmv,33367
hasta 1 smlmv,17190


In [ ]:
Credito.head(1)

,fecha_corte,tipo_credito,tipo_garantia,producto_credito,plazo_credito,tasa_efectiva_promedio_ponderada,margen_adicional,monto_desembolsado,numero_creditos_desembolsados,tipo_tasa,rango_monto_desembolsado,id_entidad,id_deudor
0,29/12/2023,credito productivo,fag,productivo rural redescuento,mas de 5 anos y hasta 7 anos,14.94,2.7,15000000.0,1,ib6,mayor a 12 smlmv y menor o igual a 25 smlmv,1,1


In [20]:
import pandas as pd

Credito['fecha_corte'] = pd.to_datetime(Credito['fecha_corte'], format='%d/%m/%Y')

In [21]:
df_municipios['departamento'] = df_municipios['departamento'].str.replace('bogota, d.c.', 'bogota', regex=False)
df_municipios['departamento'] = df_municipios['departamento'].str.replace('archipielago de san andres, providencia y santa catalina', 'islas san andres', regex=False)
df_ciiu["actividad_economica"] = df_ciiu["actividad_economica"].str.replace(".", "", regex=False)

In [ ]:
Entidades['nombre_entidad'].value_counts()

,count
nombre_entidad,
banagrario,1
rci colombia s.a.,1
banco davivienda,1
banco de bogota,1
cooperativa financiera de antioquia,1
banco de occidente,1
cotrafa,1
banco caja social s.a.,1
bancoomeva,1




```
CREATE TABLE Entidades (
    id_entidad INTEGER PRIMARY KEY,
    tipo_entidad INTEGER,
    nombre_tipo_entidad TEXT,
    codigo_entidad INTEGER,
    nombre_entidad TEXT
);

CREATE TABLE Municipio (
    codigo_municipio INTEGER PRIMARY KEY,
    municipio TEXT,
    departamento TEXT
);

CREATE TABLE Ciiu (
    codigo_ciiu TEXT PRIMARY KEY,
    actividad_economica TEXT
);

CREATE TABLE Deudor (
    id_deudor INTEGER PRIMARY KEY,
    tipo_persona TEXT,
    sexo TEXT,
    clase_deudor TEXT,
    codigo_municipio INTEGER,
    codigo_ciiu TEXT,
    grupo_etnico TEXT ,
    antiguedad_empresa TEXT,
    tamano_empresa TEXT,

    FOREIGN KEY (codigo_municipio)
        REFERENCES Municipio(codigo_municipio),

    FOREIGN KEY (codigo_ciiu)
        REFERENCES Ciiu(codigo_ciiu)
);

CREATE TABLE Credito (
    id_credito INTEGER PRIMARY KEY,
    fecha_corte TEXT,
    tipo_credito TEXT ,
    tipo_garantia TEXT,
    producto_credito TEXT ,
    plazo_credito TEXT,
    tasa_efectiva_promedio_ponderada REAL,
    margen_adicional REAL,
    monto_desembolsado INTEGER,
    numero_creditos_desembolsados INTEGER,
    tipo_tasa TEXT,
    rango_monto_desembolsado TEXT,
    id_deudor INTEGER,
    id_entidad INTEGER,

    FOREIGN KEY (id_deudor)
        REFERENCES Deudor(id_deudor),

    FOREIGN KEY (id_entidad)
        REFERENCES Entidades(id_entidad)
);

```



# Base de datos SQLite

In [4]:
import sqlite3
import pandas as pd
import numpy as np # Import numpy for np.nan

# =========================
# Conexión
# =========================
conn = sqlite3.connect("mi_base.db")
cursor = conn.cursor()

# Activar claves foráneas
cursor.execute("PRAGMA foreign_keys = ON;")
cursor.executescript("""
DROP TABLE IF EXISTS Credito;
DROP TABLE IF EXISTS Deudor;
DROP TABLE IF EXISTS Municipio;
DROP TABLE IF EXISTS Ciiu;
DROP TABLE IF EXISTS Entidades;
""")

In [5]:


# =========================
# Crear estructura
# =========================

cursor.executescript("""
CREATE TABLE Entidades (
    id_entidad INTEGER PRIMARY KEY,
    tipo_entidad INTEGER,
    nombre_tipo_entidad TEXT,
    codigo_entidad INTEGER,
    nombre_entidad TEXT
);

CREATE TABLE Municipio (
    codigo_municipio INTEGER PRIMARY KEY,
    municipio TEXT,
    departamento TEXT
);

CREATE TABLE Ciiu (
    codigo_ciiu INTEGER PRIMARY KEY,
    actividad_economica TEXT
);

CREATE TABLE Deudor (
    id_deudor INTEGER PRIMARY KEY,
    tipo_persona TEXT,
    sexo TEXT,
    clase_deudor TEXT,
    codigo_municipio INTEGER,
    codigo_ciiu INTEGER,
    grupo_etnico TEXT ,
    antiguedad_empresa TEXT,
    tamano_empresa TEXT,

    FOREIGN KEY (codigo_municipio)
        REFERENCES Municipio(codigo_municipio),

    FOREIGN KEY (codigo_ciiu)
        REFERENCES Ciiu(codigo_ciiu)
);

CREATE TABLE Credito (
    id_credito INTEGER PRIMARY KEY,
    fecha_corte TEXT,
    tipo_credito TEXT ,
    tipo_garantia TEXT,
    producto_credito TEXT ,
    plazo_credito TEXT,
    tasa_efectiva_promedio_ponderada REAL,
    margen_adicional REAL,
    monto_desembolsado INTEGER,
    numero_creditos_desembolsados INTEGER,
    tipo_tasa TEXT,
    rango_monto_desembolsado TEXT,
    id_deudor INTEGER,
    id_entidad INTEGER,

    FOREIGN KEY (id_deudor)
        REFERENCES Deudor(id_deudor),

    FOREIGN KEY (id_entidad)
        REFERENCES Entidades(id_entidad)
);
""")

conn.commit()



In [22]:
Entidades.to_sql("Entidades", conn, if_exists="append", index=False, chunksize=10000)
df_municipios.to_sql("Municipio", conn, if_exists="append", index=False, chunksize=10000)
df_ciiu.to_sql("Ciiu", conn, if_exists="append", index=False, chunksize=10000)
Deudor.to_sql("Deudor", conn, if_exists="append", index=False, chunksize=10000)

73949

In [23]:
# =========================
# Cargar datos
# =========================
Credito.to_sql("Credito", conn, if_exists="append", index=False, chunksize=10000)

conn.commit()

print("✅ Tablas creadas y datos cargados correctamente.")

# =========================
# Verificar
# =========================
for tabla in ["Entidades", "Municipio", "Ciiu", "Deudor", "Credito"]:
    count = pd.read_sql(f"SELECT COUNT(*) as total FROM {tabla}", conn)
    print(tabla, count["total"][0])

conn.close()

✅ Tablas creadas y datos cargados correctamente.
Entidades 48
Municipio 1123
Ciiu 730
Deudor 73949
Credito 680783


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("mi_base.db")

query = """
SELECT DISTINCT tipo_tasa FROM Credito EXCEPT SELECT DISTINCT c.tipo_tasa FROM Credito c JOIN Deudor d ON c.id_deudor = d.id_deudor JOIN Municipio m ON d.codigo_municipio = m.codigo_municipio WHERE m.departamento = 'cauca';
"""

df = pd.read_sql(query, conn)
print(df)

conn.close()

  tipo_tasa
0      cec$
1      dtfe
2       ipc
3      vuvr


# Fine-tunning #1

In [1]:
pip install -U transformers accelerate peft bitsandbytes trl

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

In [3]:
# Modelo base mistrall
model_name = "mistralai/Mistral-7B-v0.1"
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Import BitsAndBytesConfig
from transformers import BitsAndBytesConfig

# Configuración de cuantificación (QLoRA)
fq_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Cargar modelo en 4-bit (QLoRA)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=fq_config,
    device_map="auto"
)

# Cargar dataset
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "validation.jsonl",
        "test": "test.jsonl"
    }
)

# Formatear dataset a texto continuo
def format_example(example):
    return {
        "text": f"""{example['instruction']}

Pregunta:
{example['input']}

Respuesta:
{example['output']}"""
    }

dataset_train = dataset["train"].map(format_example)
dataset_validation = dataset["validation"].map(format_example)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [17]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,           # regularización
    bias="none",
    task_type="CAUSAL_LM",

    # opcionales pero útiles
    fan_in_fan_out=False,
    init_lora_weights="gaussian"
)
training_args = TrainingArguments(

    output_dir="./mistral-sql-test",

    # batch
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # entrenamiento
    num_train_epochs=4,
    learning_rate=1e-4,
    warmup_steps=0.05,

    # scheduler
    lr_scheduler_type="cosine",

    # regularización
    weight_decay=0.01,
    max_grad_norm=1.0,

    # estabilidad
    logging_steps=10,
    save_strategy="epoch",
    seed=42
)

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_train,
    peft_config=peft_config,
    args=training_args,
)

# Entrenar
trainer.train()

# Guardar modelo
trainer.save_model("./mistral-sql-test")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
10,0.853827
20,0.443991
30,0.353754
40,0.297269
50,0.264969
60,0.250143


In [5]:
#mistral
# Configuración LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./mistral-sql-test",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=1e-4,
    fp16=False,
    logging_steps=5,
    save_strategy="epoch"
)

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args
)

# Entrenar
trainer.train()

# Guardar modelo
trainer.save_model("./mistral-sql-test")

Step,Training Loss
5,1.271477
10,0.892249
15,0.659056
20,0.642660
25,0.548772
30,0.523036
35,0.490823
40,0.520446
45,0.471396
50,0.523022


### **After restarting the runtime, re-run the `pip install` and `import` cells.**

Then, re-initialize the tokenizer and `fq_config` for quantization, and load the saved fine-tuned model for inference:


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel # Import PeftModel

# Re-initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
tokenizer.pad_token = tokenizer.eos_token

# Re-define BitsAndBytesConfig for quantization
fq_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 1. Load the base model with quantization config
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=fq_config,
    device_map="auto",
    offload_buffers=True # Added to mitigate OOM during loading
)

# 2. Load the fine-tuned adapter and attach it to the base model
model_for_inference = PeftModel.from_pretrained(base_model, "./mistral-sql-test")

# Ensure the model is in evaluation mode
model_for_inference.eval()

print("Model loaded successfully for inference!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully for inference!


Now, you can create the text-generation pipeline using this `model_for_inference`.

In [2]:
def generar_sql(question):

    prompt = f"""
Genera la consulta SQL correcta para la siguiente pregunta.

Question:
{question}

SQL:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model_for_inference.generate(
        **inputs,
        max_new_tokens=200
    )

    sql = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return sql.split("SQL:")[-1].strip()

In [3]:
def exact_match(pred, gold):
    return int(pred.strip().lower() == gold.strip().lower())

In [4]:
import sqlite3

conn = sqlite3.connect("mi_base.db")
cursor = conn.cursor()

def execution_accuracy(pred_sql, gold_sql):

    try:
        pred = cursor.execute(pred_sql).fetchall()
        gold = cursor.execute(gold_sql).fetchall()

        return int(pred == gold)

    except:
        return 0

In [5]:
def error_rate(pred_sql):

    try:
        cursor.execute(pred_sql)
        return 0
    except:
        return 1

In [6]:
import datasets
from datasets import load_dataset
exact = 0
exec_acc = 0
errors = 0
dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "validation.jsonl",
        "test": "test.jsonl"
    }
)
test_data = dataset["test"]

for example in test_data:

    question = example["input"]
    gold_sql = example["output"]

    pred_sql = generar_sql(question)

    exact += exact_match(pred_sql, gold_sql)
    exec_acc += execution_accuracy(pred_sql, gold_sql)
    errors += error_rate(pred_sql)

n = len(test_data)

print("Total:", n)
print("Exact Match:", exact/n)
print("Execution Accuracy:", exec_acc/n)
print("Error Rate:", errors/n)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

Total: 30
Exact Match: 0.6
Execution Accuracy: 0.7666666666666667
Error Rate: 0.1


In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model_for_inference, # Use the explicitly loaded model
    tokenizer=tokenizer,
    device=0 # Specify device if not handled by device_map automatically
)

prompt = """Genera la consulta SQL correcta para la siguiente pregunta.

Pregunta:
quiero saber cual es la actividad economica mas comun en bogota

Respuesta:
"""

result = pipe(prompt, max_new_tokens=200, do_sample=False)

print(result[0]["generated_text"])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Genera la consulta SQL correcta para la siguiente pregunta.

Pregunta:
quiero saber cual es la actividad economica mas comun en bogota

Respuesta:
SELECT ci.actividad_economica, COUNT(*) AS total FROM Credito c JOIN Deudor d ON c.id_deudor = d.id_deudor JOIN Ciiu ci ON d.codigo_ciiu = ci.codigo_ciiu WHERE d.municipio = 'bogota' GROUP BY ci.actividad_economica ORDER BY total DESC LIMIT 1;


In [ ]:
import shutil

shutil.make_archive("mistral-sql-test", 'zip', "mistral-sql-test")

'/content/mistral-sql-test.zip'

In [ ]:
from google.colab import files

files.download("mistral-sql-test.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install transformers peft accelerate sentencepiece

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 82446, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 82446 (delta 43), reused 13 (delta 13), pack-reused 82347 (from 3)
Receiving objects: 100% (82446/82446), 311.42 MiB | 15.85 MiB/s, done.
Resolving deltas: 100% (59277/59277), done.
Updating files: 100% (2429/2429), done.
/content/llama.cpp


In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = "mistralai/Mistral-7B-v0.1"

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="cpu"
)

model = PeftModel.from_pretrained(
    model,
    "/content/mistral-sql-test"
)

model = model.merge_and_unload()

model.save_pretrained("/content/mistral_sql_merged")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!python convert_hf_to_gguf.py /content/mistral_sql_merged --outfile /content/mistral_sql.gguf

In [ ]:
!make

In [ ]:
!./quantize /content/mistral_sql.gguf /content/mistral_sql_q4.gguf q4_K_M